# Local-SDFT: Fine-tune a 230M model on your machine

Self-Distillation Fine-Tuning (**SDFT**) is on-policy self-distillation from demos: show the model a few gold pairs, let it answer, then LoRA-train on those self-generated targets — instead of plain **SFT** on off-policy gold text. (Shenfeld et al. study this for continual learning / less forgetting; here we run the same loop on a laptop-sized model.) **GRPO** is the on-policy RL baseline: sample groups of completions, score them, update.

Paper: [Shenfeld et al., 2026 — *Self-Distillation Enables Continual Learning*](https://self-distillation.github.io/SDFT) ([arXiv](https://arxiv.org/abs/2601.19897)).

Base model: [Liquid AI LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M) — small enough for Colab T4, Apple Silicon MPS, or even CPU smoke runs.

## Setup notes

- **Colab**: Runtime → GPU (T4 is fine). Clone the repo in the next cell, or upload the project zip and `cd` into it. The install cell uninstalls Colab's stale `torchao` so peft LoRA does not hit a version gate.
- **Local**: Apple Silicon (MPS) or CUDA preferred; CPU works for tiny smoke tests.
- Expect the working directory to be the **repo root** (`Local-SDFT/`) so `configs/compare/` and `sdft/` resolve.
- This notebook is a **smoke path** (16 train / 8 eval, GRPO capped at 4 steps). Full laptop run matching README numbers: `python scripts/run_batch1_comparison.py --num-train 32 --num-eval 16 --max-grpo-steps 16`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Install loose pins (Colab / fresh venv).
# peft>=0.19 raises ImportError if torchao is installed but <0.16 (Colab often
# ships 0.10). We don't use torchao quantization for this 230M LoRA path — peft
# falls through to the default nn.Linear dispatcher when torchao is absent.
%pip install -q "torch>=2.6" "transformers>=4.54" "trl>=0.19" "peft>=0.15" "datasets>=2.19" "accelerate>=0.33" "pyyaml>=6.0"
%pip uninstall -y torchao

REPO_URL = "https://github.com/lin826/Local-SDFT.git"
REPO_DIR = Path("Local-SDFT")

cwd = Path.cwd().resolve()
if (cwd / "sdft").is_dir() and (cwd / "configs" / "compare").is_dir():
    ROOT = cwd
    print(f"using existing repo root: {ROOT}")
elif (cwd / REPO_DIR / "sdft").is_dir():
    ROOT = (cwd / REPO_DIR).resolve()
    print(f"using cloned repo: {ROOT}")
else:
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL])
    ROOT = (cwd / REPO_DIR).resolve()
    print(f"cloned into: {ROOT}")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["PYTHONPATH"] = str(ROOT)

print("cwd:", Path.cwd())
print("PYTHONPATH:", os.environ["PYTHONPATH"])

## Three methods (same tiny data budget)

```
 alpaca slice
      |
      +---> gold targets -----------> LoRA SFT  (gold SFT)
      |
      +---> few-shot teacher gen ---> self targets -> LoRA SFT  (SDFT)
      |
      +---> prompts only -----------> GRPO rollouts + reward  (GRPO)
```

- **Gold SFT** — train on dataset `output` text as-is.
- **SDFT** — model rewrites each answer (few-shot demos), then SFT on those rewrites.
- **GRPO** — sample completion groups, score with a local instruction reward, optimize relatively.

In [ ]:
import json
from pathlib import Path

from sdft.config import load_config
from sdft.data import load_examples
from sdft.grpo_train import examples_to_grpo_jsonl

NUM_TRAIN = 16
NUM_EVAL = 8

cfg = load_config("configs/compare/batch1_sdft.yaml")
cfg.data.num_examples = NUM_TRAIN + NUM_EVAL
all_examples = load_examples(cfg.data)
train_ex = all_examples[:NUM_TRAIN]
eval_ex = all_examples[NUM_TRAIN : NUM_TRAIN + NUM_EVAL]
assert len(train_ex) == NUM_TRAIN and len(eval_ex) == NUM_EVAL

data_dir = Path("data/compare")
data_dir.mkdir(parents=True, exist_ok=True)
gold_path = data_dir / "batch1_gold.jsonl"
grpo_path = data_dir / "batch1_grpo.jsonl"

with gold_path.open("w", encoding="utf-8") as fh:
    for e in train_ex:
        fh.write(
            json.dumps(
                {
                    "prompt": e["prompt"],
                    "response": e["response"],
                    "sdft_response": e["response"],
                },
                ensure_ascii=False,
            )
            + "\n"
        )

examples_to_grpo_jsonl(train_ex, grpo_path)
print(f"wrote {gold_path} ({NUM_TRAIN} rows)")
print(f"wrote {grpo_path} ({NUM_TRAIN} rows)")
print("sample prompt:", train_ex[0]["prompt"][:120].replace("\n", " "), "…")

## SDFT teacher pass

For each training prompt, show the model a few gold demos (`num_shots=2`) and let it answer. Those self-generated strings become the SDFT targets.

If you hit CUDA/MPS OOM, drop `max_new_tokens` (e.g. 96) or `generation.batch_size` to 1 in the config / cell below.

In [ ]:
import json
from pathlib import Path

from sdft.config import load_config
from sdft.generate import generate_responses
from sdft.utils import pick_device

sdft_path = Path("data/compare/batch1_sdft.jsonl")
sdft_cfg = load_config("configs/compare/batch1_sdft.yaml")
sdft_cfg.data.num_examples = NUM_TRAIN
sdft_cfg.generation.num_shots = 2
sdft_cfg.generation.out_path = str(sdft_path)
# Safer defaults for Colab T4 / MPS smoke
sdft_cfg.generation.batch_size = 1
sdft_cfg.generation.max_new_tokens = 128

device = pick_device()
print(f"device={device}  teacher on {NUM_TRAIN} examples, num_shots=2")

try:
    gens = generate_responses(sdft_cfg, train_ex, device)
except RuntimeError as err:
    if "out of memory" in str(err).lower() or "oom" in str(err).lower():
        raise RuntimeError(
            "OOM during teacher generate. Reduce generation.max_new_tokens "
            "(try 64–96) and/or set generation.batch_size=1, then re-run this cell."
        ) from err
    raise

rows = []
for ex, gen in zip(train_ex, gens):
    if not gen:
        continue
    rows.append({"prompt": ex["prompt"], "response": ex["response"], "sdft_response": gen})

sdft_path.parent.mkdir(parents=True, exist_ok=True)
with sdft_path.open("w", encoding="utf-8") as fh:
    for row in rows:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"wrote {len(rows)}/{NUM_TRAIN} SDFT pairs -> {sdft_path}")
if rows:
    print("sample sdft_response:", rows[0]["sdft_response"][:200].replace("\n", " "), "…")

## Train three adapters (`batch_size=1` style)

Configs under `configs/compare/` already use LoRA `r=8`, `max_length=512`, and `batch_size=1` for SFT/SDFT (same vibe as the online-learning demo).

**GRPO** here is capped at `max_steps=4` for a Colab smoke. README's documented 230M table uses `--max-grpo-steps 16` via `scripts/run_batch1_comparison.py` (omit the flag for a full epoch).

In [ ]:
import subprocess
import sys
from pathlib import Path


def run_mod(*args: str) -> None:
    cmd = [sys.executable, "-m", *args]
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(Path.cwd()))


gold_path = Path("data/compare/batch1_gold.jsonl")
sdft_path = Path("data/compare/batch1_sdft.jsonl")
grpo_path = Path("data/compare/batch1_grpo.jsonl")

out_sft = Path("outputs/compare/batch1-sft-gold")
out_sdft = Path("outputs/compare/batch1-sdft")
out_grpo = Path("outputs/compare/batch1-grpo")

run_mod(
    "sdft.train",
    "--config", "configs/compare/batch1_sft_gold.yaml",
    "--data", str(gold_path),
    "--target", "gold",
    "--output-dir", str(out_sft),
)

run_mod(
    "sdft.train",
    "--config", "configs/compare/batch1_sdft.yaml",
    "--data", str(sdft_path),
    "--target", "sdft",
    "--output-dir", str(out_sdft),
)

# Colab smoke: 4 GRPO steps. Full runs: omit --max-steps or raise it.
run_mod(
    "sdft.grpo_train",
    "--config", "configs/compare/batch1_grpo.yaml",
    "--data", str(grpo_path),
    "--output-dir", str(out_grpo),
    "--max-steps", "4",
)

print("adapters:", out_sft, out_sdft, out_grpo)

## Evaluate with local instruction reward

Score **base / sft_gold / sdft / grpo** on the held-out 8 prompts using `sdft.rewards.instruction_reward` (heuristic: non-refusal + length + lexical overlap with gold). Missing adapters are skipped.

In [ ]:
import re
from pathlib import Path

import torch
from peft import PeftModel

from sdft.config import ModelConfig, load_config
from sdft.peft_utils import adapter_ready
from sdft.rewards import instruction_reward
from sdft.utils import load_model, load_tokenizer, pick_device

REFUSAL_RE = re.compile(r"(?i)\b(i('m| am) sorry|i can('t|not) (assist|help)|as an ai)\b")

MODEL_NAME = load_config("configs/compare/batch1_sdft.yaml").model.name  # LiquidAI/LFM2.5-230M
prompts = [e["prompt"] for e in eval_ex]
golds = [e["response"] for e in eval_ex]

arms = {
    "base": None,
    "sft_gold": Path("outputs/compare/batch1-sft-gold"),
    "sdft": Path("outputs/compare/batch1-sdft"),
    "grpo": Path("outputs/compare/batch1-grpo"),
}


@torch.inference_mode()
def generate_eval(adapter_dir: Path | None, max_new_tokens: int = 128) -> list[str]:
    device = pick_device()
    cfg = ModelConfig(name=MODEL_NAME)
    tokenizer = load_tokenizer(cfg)
    tokenizer.padding_side = "left"
    base = load_model(cfg, device)
    model = (
        PeftModel.from_pretrained(base, str(adapter_dir))
        if adapter_dir is not None and adapter_ready(adapter_dir)
        else base
    )
    model.eval()
    outs: list[str] = []
    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        new = out[:, enc["input_ids"].shape[1] :]
        outs.append(tokenizer.decode(new[0], skip_special_tokens=True).strip())
    del model, base
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return outs


rows = []
for name, adapter in arms.items():
    if adapter is not None and not adapter_ready(adapter):
        print(f"skip {name}: adapter missing at {adapter}")
        continue
    print(f"evaluating {name}…", flush=True)
    gens = generate_eval(adapter)
    rewards = instruction_reward(gens, gold=golds)
    refusals = sum(1 for g in gens if REFUSAL_RE.search(g))
    chars = [len(g) for g in gens]
    n = max(len(gens), 1)
    rows.append(
        {
            "arm": name,
            "mean_reward": round(sum(rewards) / n, 3),
            "refusal_rate": round(refusals / n, 3),
            "mean_chars": round(sum(chars) / n, 1),
        }
    )

print()
print("| arm | mean_reward | refusal_rate | mean_chars |")
print("|-----|-------------|--------------|------------|")
for r in rows:
    print(
        f"| {r['arm']} | {r['mean_reward']:.3f} | {r['refusal_rate']:.3f} | {r['mean_chars']:.1f} |"
    )

## What to try next

1. **Custom local JSONL** — drop Alpaca-style rows into `data/my_dataset.jsonl` and point a config at it with `dataset: json` + `data_files: ...`.
2. **Online learning web UI** — locally: `python -m web.app`, open `/data` (each message is +/0/− tone feedback on the prior reply, then a tiny LoRA SDFT update).
3. **OpenClaw tool-use** — see `docs/openclaw-rl-eval.md` for the ReTool-style loop and boxed reward.
4. **Full batch-1 comparison** (README numbers) — `python scripts/run_batch1_comparison.py --num-train 32 --num-eval 16 --max-grpo-steps 16`.

Happy tinkering — 230M is small enough that curiosity is the bottleneck, not VRAM.